In [65]:
import requests
import pandas as pd

In [66]:
COACH_SITES = {
    # 1
    "sju": {
        "school_name": "Saint Joseph's University",
        "team_name": "Hawks",
        "state": "PA",
        "coaches_url": "https://sjuhawks.com/sports/baseball/coaches",
    },

    # 2
    "villanova": {
        "school_name": "Villanova University",
        "team_name": "Wildcats",
        "state": "PA",
        "coaches_url": "https://villanova.com/sports/baseball/coaches",
    },

    # 3
    "upenn": {
        "school_name": "University of Pennsylvania",
        "team_name": "Quakers",
        "state": "PA",
        "coaches_url": "https://pennathletics.com/sports/baseball/coaches",
    },

    # 4
    "brown": {
        "school_name": "Brown University",
        "team_name": "Bears",
        "state": "RI",
        "coaches_url": "https://brownbears.com/sports/baseball/coaches",
    },

    # 5
    "fordham": {
        "school_name": "Fordham University",
        "team_name": "Rams",
        "state": "NY",
        "coaches_url": "https://fordhamsports.com/sports/baseball/coaches",
    },

    # 6
    "hofstra": {
        "school_name": "Hofstra University",
        "team_name": "Pride",
        "state": "NY",
        "coaches_url": "https://gohofstra.com/sports/baseball/coaches",
    },

    # 7
    "butler": {
        "school_name": "Butler University",
        "team_name": "Bulldogs",
        "state": "IN",
        "coaches_url": "https://butlersports.com/sports/baseball/coaches",
    },

    #8 

    "iona": {
        "school_name": "Iona University",
        "team_name": "Gaels",
        "state": "NY",
        "coaches_url": "https://icgaels.com/sports/baseball/coaches",
    },

    # 9
    "cornell": {
        "school_name": "Cornell University",
        "team_name": "Big Red",
        "state": "NY",
        "coaches_url": "https://cornellbigred.com/sports/baseball/coaches",
    },

    # 10  (domain is Boston College, not BU)
    "boston": {
        "school_name": "Boston College",
        "team_name": "Eagles",
        "state": "MA",
        "coaches_url": "https://bceagles.com/sports/baseball/coaches",
    },

    # 11
    "princeton": {
        "school_name": "Princeton University",
        "team_name": "Tigers",
        "state": "NJ",
        "coaches_url": "https://goprincetontigers.com/coaches.aspx?path=baseball",
    },

    # 12
    "geogetown": {
        "school_name": "Georgetown University",
        "team_name": "Hoyas",
        "state": "DC",
        "coaches_url": "https://guhoyas.com/sports/baseball/coaches",
    },

    # 13  (this URL is NC State → Wolfpack)
    "north_carolina": {
        "school_name": "North Carolina State University",
        "team_name": "Wolfpack",
        "state": "NC",
        "coaches_url": "https://gopack.com/sports/baseball/coaches",
    },

    # 14
    "georgia": {
        "school_name": "University of Georgia",
        "team_name": "Bulldogs",
        "state": "GA",
        "coaches_url": "https://georgiadogs.com/sports/baseball/coaches",
    },

    # 15
    "elon": {
        "school_name": "Elon University",
        "team_name": " Phoenix",
        "state": "NC",
        "coaches_url": "https://elonphoenix.com/sports/baseball/coaches",
    },

    # 16
    "monmouth": {
        "school_name": "Monmouth University",
        "team_name": "Hawks",
        "state": "NJ",
        "coaches_url": "https://monmouthhawks.com/sports/baseball/coaches",
    },

    # 17
    "ball": {
        "school_name": "Ball State University",
        "team_name": "Cardinals",
        "state": "IN",
        "coaches_url": "https://ballstatesports.com/sports/baseball/coaches",
    },

    # 18  (key name stays 'bucknel' to match your players SITES)
    "bucknel": {
        "school_name": "Bucknell University",
        "team_name": "Bison",
        "state": "PA",
        "coaches_url": "https://bucknellbison.com/sports/baseball/coaches",
    },

    # 19  (key name stays 'rudgers' to match your players SITES)
    "rudgers": {
        "school_name": "Rutgers University",
        "team_name": "Scarlet Knights",
        "state": "NJ",
        "coaches_url": "https://scarletknights.com/sports/baseball/coaches",
    },

    # 20
    "gannon": {
        "school_name": "Gannon University",
        "team_name": "Golden Knights",
        "state": "PA",
        "coaches_url": "https://gannonsports.com/sports/baseball/coaches",
    },
}

In [67]:
def fetch_html(url: str) -> str:
    """
    Download HTML from a URL with a simple User-Agent.
    """
    headers = {
        "User-Agent": "Mozilla/5.0 (compatible; CoachesScraper/1.0)"
    }
    resp = requests.get(url, headers=headers, timeout=20)
    resp.raise_for_status()
    return resp.text


def pick_main_table(tables: list[pd.DataFrame]) -> pd.DataFrame | None:
    """
    Given tables from pd.read_html, pick the 'main' one.
    Heuristic: largest table by rows*cols, ignoring tiny tables.
    """
    if not tables:
        return None

    best_df = None
    best_size = -1

    for df in tables:
        rows, cols = df.shape
        # ignore trivial tables like menus, 1-row tables, etc.
        if rows < 2 or cols < 2:
            continue
        size = rows * cols
        if size > best_size:
            best_size = size
            best_df = df

    return best_df


def extract_all_staff_tables(url: str) -> pd.DataFrame:
    """
    Fetch URL, read ALL usable HTML tables, and stack them into
    one DataFrame. This will include separate Coaching Staff and
    Support Staff tables if they exist.

    If column names differ between tables, pandas will create new
    columns when we concat (with NaN where missing).
    """
    html = fetch_html(url)

    try:
        tables = pd.read_html(html)
    except ValueError:
        raise RuntimeError("No HTML tables found on this page.")

    staff_tables = []
    for idx, df in enumerate(tables):
        rows, cols = df.shape
        # skip trivial stuff (like 1-row header or layout tables)
        if rows < 2 or cols < 2:
            continue

        df = df.copy()
        df.columns = [str(c).strip() for c in df.columns]
        df["table_index"] = idx  # optional: which table it came from
        staff_tables.append(df)

    if not staff_tables:
        raise RuntimeError("No suitable staff tables found on this page.")

    combined = pd.concat(staff_tables, ignore_index=True, sort=False)
    return combined



In [68]:
def process_coach_site(site_key: str) -> pd.DataFrame:
    """
    For one school:
      - fetch the coaches table
      - keep ALL original columns
      - add school_name, team_name, state, school_key
    """
    key = site_key.strip().lower()
    if key not in COACH_SITES:
        raise ValueError(f"Unknown site key '{site_key}'. Known: {list(COACH_SITES.keys())}")

    cfg = COACH_SITES[key]
    url = cfg["coaches_url"]

    print(f"\n=== Processing coaches for {key} ({cfg['school_name']}) ===")
    print("URL:", url)

    df_raw = extract_all_staff_tables(url)
    print("Raw shape:", df_raw.shape)
    print("Columns:", list(df_raw.columns))

    df = df_raw.copy()
    df["school_name"] = cfg["school_name"]
    df["team_name"] = cfg["team_name"]
    df["state"] = cfg["state"]

    return df




In [ ]:
def normalize_coach_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    - Merge Position & Title into 'position'
    - Merge Email & Email Address into 'email'
    - Drop instagram/twitter-related columns
    """
    out = df.copy()

    # --- 1) Rename known variants to canonical names ---
    rename_map = {}

    for col in df.columns:
        cl = col.strip().lower()

    
        if cl == "title" or cl == "position":
            rename_map[col] = "Position"

    
        elif cl == "email" or cl == "email address":
            rename_map[col] = "Email Address"


    if rename_map:
        out = out.rename(columns=rename_map)

    # --- 2) Collapse duplicate columns ('Position' + 'Title' -> 'position')
    if out.columns.duplicated().any():
        new_df = pd.DataFrame(index=out.index)
        for col in dict.fromkeys(out.columns):  # preserve order
            subset = out.loc[:, out.columns == col]
            if subset.shape[1] == 1:
                new_df[col] = subset.iloc[:, 0]
            else:
                # take first non-null across duplicates
                combined = subset.bfill(axis=1).iloc[:, 0]
                new_df[col] = combined
        out = new_df

    # --- 3) Drop instagram / twitter columns ---
    drop_cols = [
        c for c in out.columns
        if ("instagram" in c.lower()) or ("twitter" in c.lower()) or ("location" in c.lower()) or ("image" in c.lower()) or ("admin" in c.lower()) or ("x" in c.lower()) or ("phone" in c.lower())
    ]
    if drop_cols:
        out = out.drop(columns=drop_cols)

    return out


In [ ]:
coach_dfs = []

for key in COACH_SITES.keys():
    try:
        df_one = process_coach_site(key)   # your existing function
        df_one = normalize_coach_columns(df_one)  #  apply cleanup here
        coach_dfs.append(df_one)
    except Exception as e:
        print(f"!! Skipping {key} due to error: {e}")

if coach_dfs:
    df_coaches_all = pd.concat(coach_dfs, ignore_index=True, sort=False)
    print("\nCombined coaches shape:", df_coaches_all.shape)
    print("Combined columns:", list(df_coaches_all.columns))

    df_coaches_all.to_csv("coaches.csv", index=False)
    print("Saved coaches.csv")
else:
    print("No coaches tables were successfully processed.")



=== Processing coaches for sju (Saint Joseph's University) ===
URL: https://sjuhawks.com/sports/baseball/coaches


/var/folders/bf/sbry5wts551d8l1lfbz4y6br0000gn/T/ipykernel_58217/3789480318.py:49: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(html)


Raw shape: (13, 5)
Columns: ['Name', 'Position', 'Phone', 'Email', 'table_index']

=== Processing coaches for villanova (Villanova University) ===
URL: https://villanova.com/sports/baseball/coaches


/var/folders/bf/sbry5wts551d8l1lfbz4y6br0000gn/T/ipykernel_58217/3789480318.py:49: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(html)


Raw shape: (8, 5)
Columns: ['Name', 'Title', 'Phone', 'Email Address', 'table_index']

=== Processing coaches for upenn (University of Pennsylvania) ===
URL: https://pennathletics.com/sports/baseball/coaches


...

/var/folders/bf/sbry5wts551d8l1lfbz4y6br0000gn/T/ipykernel_58217/3789480318.py:49: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(html)


Raw shape: (9, 7)
Columns: ['Image', 'Name', 'Title', 'Phone', 'Email Address', 'Twitter', 'table_index']

=== Processing coaches for bucknel (Bucknell University) ===
URL: https://bucknellbison.com/sports/baseball/coaches


/var/folders/bf/sbry5wts551d8l1lfbz4y6br0000gn/T/ipykernel_58217/3789480318.py:49: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(html)
/var/folders/bf/sbry5wts551d8l1lfbz4y6br0000gn/T/ipykernel_58217/3789480318.py:49: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(html)


Raw shape: (4, 5)
Columns: ['Name', 'Title', 'Email Address', 'Phone', 'table_index']

=== Processing coaches for rudgers (Rutgers University) ===
URL: https://scarletknights.com/sports/baseball/coaches
Raw shape: (18, 7)
Columns: ['Name', 'Title', 'Email Address', 'Phone', 'Twitter', 'Admin', 'table_index']

=== Processing coaches for gannon (Gannon University) ===
URL: https://gannonsports.com/sports/baseball/coaches
Raw shape: (3, 5)
Columns: ['Name', 'Title', 'Email Address', 'Phone', 'table_index']

Combined coaches shape: (194, 6)
Combined columns: ['Name', 'Position', 'Email Address', 'school_name', 'team_name', 'state']
Saved coaches.csv


/var/folders/bf/sbry5wts551d8l1lfbz4y6br0000gn/T/ipykernel_58217/3789480318.py:49: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(html)


In [71]:
def build_school_table(csv_path: str = "school.csv") -> pd.DataFrame:
    """
    Build a table with ONE row per school:
      - school_name
      - team_name
      - state
    (No school_key column in the CSV.)
    """
    rows = []
    for key, cfg in COACH_SITES.items():
        rows.append(
            {
                "school_name": cfg["school_name"],
                "team_name": cfg["team_name"],
                "state": cfg["state"],
            }
        )

    df_schools = (
        pd.DataFrame(rows)
        .drop_duplicates()
        .sort_values("school_name")
        .reset_index(drop=True)
    )

    df_schools.to_csv(csv_path, index=False)
    print(f"Saved {len(df_schools)} schools to {csv_path}")
    return df_schools


In [72]:
df_schools = build_school_table()
df_schools.head()


Saved 20 schools to school.csv


,school_name,team_name,state
0,Ball State University,Cardinals,IN
1,Boston College,Eagles,MA
2,Brown University,Bears,RI
3,Bucknell University,Bison,PA
4,Butler University,Bulldogs,IN
